Mount google drive:

In [ ]:
import os
from google.colab import drive

def mount_drive_safely():
    """
    Safely mounts Google Drive in Google Colab.
    - Skips mounting if already mounted
    - Verifies mount was successful after mounting
    - Prints clear status messages
    """
    mount_point = "/content/drive"

    # ── Check if already mounted ───────────────────────────────
    if os.path.isdir(os.path.join(mount_point, "MyDrive")):
        print("✔ Google Drive is already mounted.")
        print(f"  Mount point : {mount_point}")
        print(f"  MyDrive     : {os.path.join(mount_point, 'MyDrive')}")
        return True

    # ── Attempt mount ──────────────────────────────────────────
    print("Mounting Google Drive...")
    try:
        drive.mount(mount_point, force_remount=False)
    except Exception as e:
        print(f"✘ Mount failed: {e}")
        return False

    # ── Verify mount succeeded ─────────────────────────────────
    if os.path.isdir(os.path.join(mount_point, "MyDrive")):
        print("✔ Google Drive mounted successfully.")
        print(f"  Mount point : {mount_point}")
        return True
    else:
        print("✘ Mount appeared to succeed but MyDrive is not accessible.")
        return False


# ── Run it ─────────────────────────────────────────────────────
mounted = mount_drive_safely()

if not mounted:
    raise RuntimeError(
        "Google Drive could not be mounted. "
        "Please run this cell again or restart the runtime."

Our video classification model:

In [3]:
import os
import gc
import cv2
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from google.colab import drive
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# ===============================
# GPU CHECK
# ===============================
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# ===============================
# CONFIG
# ===============================
TRAIN_DIR = "/content/drive/My Drive/Larynx/trainset"
TEST_DIR = "/content/drive/My Drive/Larynx/Apollodata"

BATCH_SIZE = 16
EPOCHS = 40
LR = 1e-4

#Vary the number of words you want to classify here
WORDS = ["tomorrow", "yesterday", "banana"]
WORD_TO_IDX = {w: i for i, w in enumerate(WORDS)}


#Vary temporal resolution (downsampling) here
FIXED_FRAME_COUNT = 16

#Vary spatial resolution (resize) here
IMAGE_SIZE = (64, 48)   # (width, height)

# ===============================
# SAVE CONFIG
# ===============================
SAVE_ROOT = "/content/drive/My Drive/experiments/resizeanalysis_single_64x48"
os.makedirs(SAVE_ROOT, exist_ok=True)

# ===============================
# AUGMENTATION SETTINGS
# ===============================
AUG_PROB = 0.90
ROTATION_RANGE = (-8, 8)
TRANSLATION_X_RANGE = (-10, 10)
TRANSLATION_Y_RANGE = (-8, 8)
SCALE_RANGE = (0.95, 1.05)
NOISE_STD_RANGE = (0.0, 0.02)

# ===============================
# HELPER — DRIVE & PATH CHECKS
# ===============================
def check_drive_and_paths(train_dir, test_dir):
    if not os.path.exists("/content/drive"):
        print("Mounting Google Drive...")
        drive.mount("/content/drive")

    if not os.path.exists("/content/drive"):
        print("ERROR: Google Drive could not be mounted.")
        return False

    my_drive_root = "/content/drive/My Drive"
    if not os.path.exists(my_drive_root):
        print(f"ERROR: '{my_drive_root}' does not exist.")
        return False

    print("\nChecking directory paths:")
    common_parent_dir = os.path.dirname(train_dir)

    if not os.path.exists(common_parent_dir):
        print(f"ERROR: The base directory '{common_parent_dir}' was not found.")
        try:
            items_in_my_drive = [f for f in os.listdir(my_drive_root) if not f.startswith('.')]
            print(items_in_my_drive[:20])
            if len(items_in_my_drive) > 20:
                print("... (and more)")
        except Exception as e:
            print(f"Could not list contents of '{my_drive_root}': {e}")
        return False
    elif not os.path.isdir(common_parent_dir):
        print(f"ERROR: '{common_parent_dir}' exists but is not a directory.")
        return False
    else:
        print(f"'{common_parent_dir}' found. Contents: {os.listdir(common_parent_dir)}")

    if not os.path.exists(train_dir):
        print(f"ERROR: The training directory '{train_dir}' was not found.")
        return False
    if not os.path.isdir(train_dir):
        print(f"ERROR: '{train_dir}' exists but is not a directory.")
        return False

    if not os.path.exists(test_dir):
        print(f"ERROR: The testing directory '{test_dir}' was not found.")
        return False
    if not os.path.isdir(test_dir):
        print(f"ERROR: '{test_dir}' exists but is not a directory.")
        return False

    print("SUCCESS: All required directories exist.")
    return True

# ===============================
# LOAD VIDEO (ALL FRAMES)
# ===============================
def load_video(path, img_width, img_height):
    frames = []

    if path.endswith(".npy"):
        arr = np.load(path)
        if len(arr.shape) == 4:
            arr = arr.squeeze(-1)

        for frame in arr:
            frame = cv2.resize(frame, (img_width, img_height))
            frames.append(frame)

    else:
        cap = cv2.VideoCapture(path)
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, (img_width, img_height))
            frames.append(frame)
        cap.release()

    if len(frames) == 0:
        raise ValueError(f"No frames in {path}")

    frames = np.array(frames, dtype=np.float32) / 255.0
    return frames  # [T, H, W]

# ===============================
# UNIFORM TEMPORAL SAMPLING
# ===============================
def sample_fixed_number_of_frames(frames_np, target_frames):
    T_total = len(frames_np)

    if T_total == 0:
        raise ValueError("No frames available for sampling.")

    indices = np.linspace(0, T_total - 1, num=target_frames)
    indices = np.round(indices).astype(int)
    sampled = frames_np[indices]
    return sampled



# ===============================
# DATASET
# ===============================
class VideoDataset(Dataset):
    def __init__(self, root_dir, target_frames, img_size, training=True):
        self.samples = []
        self.labels = []
        self.target_frames = target_frames
        self.training = training
        self.img_width, self.img_height = img_size

        if not os.path.exists(root_dir) or not os.path.isdir(root_dir):
            raise ValueError(f"Directory not found: {root_dir}")

        for folder in sorted(os.listdir(root_dir)):
            folder_path = os.path.join(root_dir, folder)
            if not os.path.isdir(folder_path):
                continue

            word = next((w for w in WORDS if w in folder.lower()), None)
            if word is None:
                continue

            for file in os.listdir(folder_path):
                if file.lower().endswith((".avi", ".mp4", ".mov", ".mkv", ".npy")):
                    full_path = os.path.join(folder_path, file)
                    self.samples.append(full_path)
                    self.labels.append(WORD_TO_IDX[word])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        label = self.labels[idx]

        frames = load_video(path, self.img_width, self.img_height)
        frames = sample_fixed_number_of_frames(frames, self.target_frames)

        frames_tensor = torch.from_numpy(frames)  # [T, H, W]

        if self.training and random.random() < AUG_PROB:
            angle = random.uniform(*ROTATION_RANGE)
            translations = (
                random.randint(*TRANSLATION_X_RANGE),
                random.randint(*TRANSLATION_Y_RANGE)
            )
            scale = random.uniform(*SCALE_RANGE)
            noise_sigma = random.uniform(*NOISE_STD_RANGE)

            transformed_frames = []
            for i in range(frames_tensor.shape[0]):
                frame = frames_tensor[i].unsqueeze(0)  # [1, H, W]

                frame = TF.affine(
                    frame,
                    angle=angle,
                    translate=translations,
                    scale=scale,
                    shear=0,
                    interpolation=T.InterpolationMode.BILINEAR
                )

                if noise_sigma > 0:
                    frame = frame + torch.randn_like(frame) * noise_sigma

                frame = torch.clamp(frame, 0.0, 1.0)
                transformed_frames.append(frame)

            processed_sequence = torch.stack(transformed_frames)  # [T, 1, H, W]
        else:
            processed_sequence = frames_tensor.unsqueeze(1)  # [T, 1, H, W]

        true_len = self.target_frames

        return (
            processed_sequence,
            torch.tensor(label, dtype=torch.long),
            torch.tensor(true_len, dtype=torch.long)
        )

# ===============================
# MODEL
# ===============================
class FrameEncoder(nn.Module):
    def __init__(self, img_height, img_width, latent_dim=64):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1),
            nn.GroupNorm(4, 8),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, 3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, img_height, img_width)
            out = self.conv(dummy)
            self.flat = out.view(1, -1).shape[1]

        self.fc = nn.Linear(self.flat, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class VideoModel(nn.Module):
    def __init__(self, img_height, img_width):
        super().__init__()

        self.encoder = FrameEncoder(
            img_height=img_height,
            img_width=img_width,
            latent_dim=64
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=32,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, len(WORDS))
        )

    def forward(self, x, lengths):
        B, T_seq, C, H, W = x.shape

        encoded_seq = self.encoder(x.view(B * T_seq, C, H, W)).view(B, T_seq, -1)

        packed = pack_padded_sequence(
            encoded_seq,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        lstm_output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=T_seq
        )

        mask = (torch.arange(T_seq, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)).float()
        mask = mask.unsqueeze(-1)

        summed = (lstm_output * mask).sum(dim=1)
        denom = lengths.clamp(min=1).unsqueeze(1).float()
        pooled_output = summed / denom

        pooled_output = self.dropout(pooled_output)
        return self.fc(pooled_output)

# ===============================
# TRAIN
# ===============================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for sequence, y, lengths in loader:
        sequence = sequence.to(device)
        y = y.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()
        out = model(sequence, lengths)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(out.data, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy

# ===============================
# EVALUATION
# ===============================
def evaluate(model, loader, criterion, target_names, print_report=True):
    model.eval()
    preds = []
    labels = []
    total_loss = 0.0

    with torch.no_grad():
        for sequence, y, lengths in loader:
            sequence = sequence.to(device)
            y = y.to(device)
            lengths = lengths.to(device)

            output = model(sequence, lengths)
            loss = criterion(output, y)
            total_loss += loss.item()

            _, predicted = torch.max(output.data, 1)
            preds.extend(predicted.cpu().numpy())
            labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * np.mean(np.array(preds) == np.array(labels))

    if print_report:
        print("\n" + "=" * 60)
        print("Classification Report:")
        print("=" * 60)
        print(classification_report(labels, preds, target_names=target_names))

        print(f"\nOverall Test Loss: {avg_loss:.4f}")
        print(f"Overall Test Accuracy: {accuracy:.2f}%")

    return avg_loss, accuracy

# ===============================
# EXPERIMENT RUNNER
# ===============================
def run_experiment(img_size):
    img_width, img_height = img_size
    experiment_name = f"resize_{img_width}x{img_height}"
    save_dir = os.path.join(SAVE_ROOT, experiment_name)
    os.makedirs(save_dir, exist_ok=True)

    best_model_path = os.path.join(save_dir, "best_model.pth")
    history_path = os.path.join(save_dir, "training_history.npy")
    summary_path = os.path.join(save_dir, "summary.json")
    plot_loss_path = os.path.join(save_dir, "loss_plot.png")
    plot_acc_path = os.path.join(save_dir, "accuracy_plot.png")

    print("\n" + "#" * 80)
    print(f"RUNNING EXPERIMENT: {experiment_name}")
    print("#" * 80)

    train_ds = VideoDataset(
        TRAIN_DIR,
        target_frames=FIXED_FRAME_COUNT,
        img_size=img_size,
        training=True
    )
    test_ds = VideoDataset(
        TEST_DIR,
        target_frames=FIXED_FRAME_COUNT,
        img_size=img_size,
        training=False
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = VideoModel(img_height=img_height, img_width=img_width).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=3e-4)
    criterion = nn.CrossEntropyLoss()

    best_test_loss = float("inf")
    best_epoch = -1

    epoch_list = []
    train_loss_history = []
    test_loss_history = []
    train_acc_history = []
    test_acc_history = []

    for epoch in range(EPOCHS):
        print("\n" + "=" * 60)
        print(f"[{experiment_name}] Epoch {epoch + 1}/{EPOCHS}")
        print("=" * 60)

        train_loss, train_accuracy = train_epoch(model, train_loader, optimizer, criterion)
        test_loss, test_accuracy = evaluate(model, test_loader, criterion, WORDS, print_report=False)

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Accuracy: {train_accuracy:.2f}%")
        print(f"Test Loss: {test_loss:.4f}")
        print(f"Test Accuracy: {test_accuracy:.2f}%")

        epoch_list.append(epoch + 1)
        train_loss_history.append(train_loss)
        test_loss_history.append(test_loss)
        train_acc_history.append(train_accuracy)
        test_acc_history.append(test_accuracy)

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved at epoch {best_epoch} with test loss {best_test_loss:.4f}")

    print("\nTraining complete.")
    print(f"Best epoch: {best_epoch}")
    print(f"Best saved test loss: {best_test_loss:.4f}")

    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        print(f"Loaded best model from '{best_model_path}'")

    print("\nFinal Evaluation on Test Set (Best Model)")
    final_test_loss, final_test_accuracy = evaluate(model, test_loader, criterion, WORDS, print_report=True)

    history = {
        "experiment_name": experiment_name,
        "image_size": {
            "width": img_width,
            "height": img_height
        },
        "fixed_frame_count": FIXED_FRAME_COUNT,
        "epoch_list": epoch_list,
        "train_loss_history": train_loss_history,
        "test_loss_history": test_loss_history,
        "train_acc_history": train_acc_history,
        "test_acc_history": test_acc_history,
        "best_epoch": best_epoch,
        "best_test_loss": best_test_loss,
        "final_test_loss": final_test_loss,
        "final_test_accuracy": final_test_accuracy,
        "config": {
            "BATCH_SIZE": BATCH_SIZE,
            "EPOCHS": EPOCHS,
            "LR": LR,
            "WORDS": WORDS,
            "AUG_PROB": AUG_PROB,
            "ROTATION_RANGE": ROTATION_RANGE,
            "TRANSLATION_X_RANGE": TRANSLATION_X_RANGE,
            "TRANSLATION_Y_RANGE": TRANSLATION_Y_RANGE,
            "SCALE_RANGE": SCALE_RANGE,
            "NOISE_STD_RANGE": NOISE_STD_RANGE
        }
    }

    np.save(history_path, history, allow_pickle=True)
    with open(summary_path, "w") as f:
        json.dump(history, f, indent=4)

    # loss plot
    plt.figure(figsize=(7, 5))
    plt.plot(epoch_list, train_loss_history, marker='o', label='Train Loss')
    plt.plot(epoch_list, test_loss_history, marker='s', label='Test Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss Curves - {img_width}x{img_height}")
    plt.xticks(epoch_list)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_loss_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    #accuracy plot
    plt.figure(figsize=(7, 5))
    plt.plot(epoch_list, train_acc_history, marker='o', label='Train Accuracy')
    plt.plot(epoch_list, test_acc_history, marker='s', label='Test Accuracy')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Accuracy Curves - {img_width}x{img_height}")
    plt.xticks(epoch_list)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_acc_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    return history

# ===============================
# MAIN
# ===============================
if __name__ == "__main__":
    if not check_drive_and_paths(TRAIN_DIR, TEST_DIR):
        print("Exiting due to path issues. Please correct the directory paths in TRAIN_DIR and TEST_DIR.")
    else:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        history = run_experiment(IMAGE_SIZE)

        summary_all_path = os.path.join(SAVE_ROOT, "all_runs_summary.json")
        compact_summary = {
            f"{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}": {
                "best_epoch": history["best_epoch"],
                "best_test_loss": history["best_test_loss"],
                "final_test_loss": history["final_test_loss"],
                "final_test_accuracy": history["final_test_accuracy"]
            }
        }

        with open(summary_all_path, "w") as f:
            json.dump(compact_summary, f, indent=4)

        print("\nExperiment completed.")
        print(f"Saved everything under: {SAVE_ROOT}")
        print("Files include:")
        print(f"- per-run folder: resize_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}")
        print("- loss_plot.png")
        print("- accuracy_plot.png")
        print("- best_model.pth")
        print("- training_history.npy")
        print("- summary.json")
        print("- all_runs_summary.json")

PyTorch version: 2.10.0+cpu
CUDA available: False
Using device: cpu

Checking directory paths:
ERROR: The base directory '/content/drive/My Drive/Larynx' was not found.
['experiments']
Exiting due to path issues. Please correct the directory paths in TRAIN_DIR and TEST_DIR.


Interspeaker 2D CNN classification model (a random frame is extracted from each video, and interspeaker classification is conducted to analyse the effect of inter-speaker strcutural differenes in a machine learning perspective):

In [ ]:
import os
import gc
import re
import cv2
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import drive
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# ===============================
# GPU CHECK
# ===============================
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# ===============================
# CONFIG
# ===============================

# ── Root directories ───────────────────────────────────────────

TRAIN_ROOT = "/content/drive/MyDrive/Task-camera-6" #dataset2- banana, yesterday, tomorrow (silent speech)
TEST_ROOT  = "/content/drive/MyDrive/Task-camera-6.1" #dataset1- engineering and sports (voiced speech)

# ── Multi-root: both trainset + testset subfolders are scanned ─
# Code's training data  = TRAIN_ROOT/trainset + TRAIN_ROOT/testset
# Code's testing data   = TEST_ROOT/trainset  + TEST_ROOT/testset
TRAIN_DIRS = [
    os.path.join(TRAIN_ROOT, "Trainset"),
    os.path.join(TRAIN_ROOT, "Testset"),
]
TEST_DIRS = [
    os.path.join(TEST_ROOT, "Trainset"),
    os.path.join(TEST_ROOT, "Testset"),
]

IMG_HEIGHT = 192
IMG_WIDTH  = 256

BATCH_SIZE   = 16
EPOCHS       = 40
LR           = 1e-4
WEIGHT_DECAY = 3e-4

SAVE_ROOT = "/content/drive/MyDrive/experiments/speakerclassification"
os.makedirs(SAVE_ROOT, exist_ok=True)

VIDEO_EXTENSIONS = (".mp4",)

TRAIN_VIDEOS_PER_SUBJECT  = None     # None = use all
TEST_VIDEOS_PER_SUBJECT   = 15
TEST_VIDEO_SELECTION_MODE = "first"  # "first" or "random"
RANDOM_SEED               = 42

EARLY_STOPPING_PATIENCE   = 8

# ===============================
# AUGMENTATION SETTINGS
# ===============================
AUG_PROB         = 0.8
ROTATION_RANGE   = (-5,   5)
SCALE_RANGE      = (0.95, 1.05)
BRIGHTNESS_RANGE = (0.90, 1.10)
CONTRAST_RANGE   = (0.80, 1.20)
NOISE_STD_RANGE  = (0.0,  0.02)

# ===============================
# HELPERS
# ===============================
def mount_and_check(train_dirs, test_dirs):
    if not os.path.exists("/content/drive"):
        print("Mounting Google Drive...")
        drive.mount("/content/drive")

    ok = True
    for label, dirs in [("Train", train_dirs), ("Test", test_dirs)]:
        for path in dirs:
            if not os.path.exists(path):
                print(f"ERROR: {label} directory not found: {path}")
                ok = False
            else:
                contents = os.listdir(path)
                print(f"SUCCESS: {label} dir found → {path}")
                print(f"  Contents ({len(contents)} items): {contents[:10]}")
    return ok


def extract_subject_name(folder_name):
    """
    Extracts subject ID from folder names like:
      Subj1-engineering → subj1
      Subj2-sports      → subj2
    Returns lowercase string like 'subj1', or None if not matched.
    """
    folder_name = folder_name.strip()
    match = re.search(r"subj\s*(\d+)", folder_name, flags=re.IGNORECASE)
    if match:
        return f"subj{match.group(1)}"
    return None


def load_video_frames(path):
    """
    Loads all frames from one mp4 video.
    Pipeline:
      1. Read frame (BGR)
      2. Convert to grayscale
      3. Resize to (IMG_WIDTH, IMG_HEIGHT)
      4. Apply CLAHE — normalises local contrast, corrects dark/bright videos
      5. Normalise to float32 [0, 1]
    Returns np.array [T, H, W] float32 in [0, 1].
    """
    frames = []
    cap    = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {path}")

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (IMG_WIDTH, IMG_HEIGHT))
        frame = clahe.apply(frame)                        # CLAHE on uint8
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError(f"No frames found in {path}")

    return np.array(frames, dtype=np.float32) / 255.0    # normalise after CLAHE


# ===============================
# DATASET
# ===============================
class SpeakerRandomFrameDataset(Dataset):
    """
    Accepts a LIST of root directories (root_dirs).
    Scans every directory in the list for subject subfolders.
    All videos found across all directories are merged per subject
    before any video limit is applied.

    Example:
        root_dirs = [".../trainset", ".../testset"]
        Both folders are scanned and merged into one flat dataset.

    Train : random frame per video, with augmentation.
    Test  : middle frame per video, no augmentation.
    """

    def __init__(
        self,
        root_dirs,
        subject_to_idx=None,
        training=True,
        videos_per_subject=None,
        video_selection_mode="first",
        random_seed=42,
    ):

        if isinstance(root_dirs, str):
            root_dirs = [root_dirs]

        self.root_dirs     = root_dirs
        self.training      = training
        self.samples       = []
        self.labels        = []
        self.subject_names = []

        # Validate all directories exist
        for root_dir in root_dirs:
            if not os.path.isdir(root_dir):
                raise ValueError(f"Directory not found: {root_dir}")

        rng = random.Random(random_seed)

        # ── Step 1: scan ALL root_dirs, merge videos per subject ───────
        # subject_videos: { "subj1": [path,...], "subj2": [...],... }
        subject_videos: dict[str, list[str]] = {}

        for root_dir in root_dirs:
            print(f"\n  Scanning: {root_dir}")

            for folder in sorted(os.listdir(root_dir)):
                folder_path = os.path.join(root_dir, folder)
                if not os.path.isdir(folder_path):
                    continue

                subject_name = extract_subject_name(folder)
                if subject_name is None:
                    print(f"    [SKIP] Unrecognised folder pattern: '{folder}'")
                    continue

                video_files = sorted([
                    os.path.join(folder_path, f)
                    for f in os.listdir(folder_path)
                    if f.lower().endswith(VIDEO_EXTENSIONS)
                ])

                if len(video_files) == 0:
                    print(f"    [SKIP] No.mp4 files in: '{folder}'")
                    continue

                if subject_name not in subject_videos:
                    subject_videos[subject_name] = []

                # Merge videos from this folder into the subject's pool
                subject_videos[subject_name].extend(video_files)
                print(f"    '{folder}' → subject='{subject_name}' "
                      f"({len(video_files)} videos)")

        # ── Step 2: build subject_to_idx ──────────────────────────────
        if subject_to_idx is None:
            self.subject_to_idx = {
                s: i for i, s in enumerate(sorted(subject_videos.keys()))
            }
        else:
            self.subject_to_idx = subject_to_idx

        self.idx_to_subject = {v: k for k, v in self.subject_to_idx.items()}

        # ── Step 3: apply per-subject video limit & build flat lists ───
        for subject_name, all_paths in sorted(subject_videos.items()):
            if subject_name not in self.subject_to_idx:
                print(f"  [SKIP] Subject '{subject_name}' not in subject_to_idx.")
                continue

            label     = self.subject_to_idx[subject_name]
            all_paths = sorted(set(all_paths))   # deduplicate

            if videos_per_subject is not None:
                if len(all_paths) < videos_per_subject:
                    print(
                        f"  [WARN] '{subject_name}' has only {len(all_paths)} "
                        f"videos (requested {videos_per_subject})"
                    )
                if video_selection_mode == "random":
                    chosen = rng.sample(
                        all_paths, min(len(all_paths), videos_per_subject)
                    )
                    chosen = sorted(chosen)
                else:
                    chosen = all_paths[:videos_per_subject]
            else:
                chosen = all_paths

            for path in chosen:
                self.samples.append(path)
                self.labels.append(label)
                self.subject_names.append(subject_name)

        # ── Step 4: report ─────────────────────────────────────────────
        subject_count = {}
        for s in self.subject_names:
            subject_count[s] = subject_count.get(s, 0) + 1

        print(f"\nDataset roots : {self.root_dirs}")
        print(f"Subjects      : {sorted(self.subject_to_idx.keys())}")
        print(f"Total videos  : {len(self.samples)}")
        print("Videos per subject:")
        for s in sorted(subject_count.keys()):
            print(f"  {s}: {subject_count[s]}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path  = self.samples[idx]
        label = self.labels[idx]

        frames = load_video_frames(path)   # [T, H, W] float32 [0,1]

        # ── Frame selection ────────────────────────────────────────────
        if self.training:
            frame_idx = random.randint(0, len(frames) - 1)
        else:
            frame_idx = len(frames) // 2

        frame        = frames[frame_idx]
        frame_tensor = torch.from_numpy(frame).unsqueeze(0)   # [1, H, W]

        # ── Augmentation (training only) ───────────────────────────────
        if self.training and random.random() < AUG_PROB:
            angle      = random.uniform(*ROTATION_RANGE)
            scale      = random.uniform(*SCALE_RANGE)
            noise_std  = random.uniform(*NOISE_STD_RANGE)
            brightness = random.uniform(*BRIGHTNESS_RANGE)
            contrast   = random.uniform(*CONTRAST_RANGE)

            # translate=(0,0) — translation removed
            frame_tensor = TF.affine(
                frame_tensor,
                angle=angle,
                translate=(0, 0),
                scale=scale,
                shear=0,
                interpolation=T.InterpolationMode.BILINEAR,
            )

            frame_tensor = TF.adjust_brightness(frame_tensor, brightness)
            frame_tensor = TF.adjust_contrast(frame_tensor, contrast)

            if noise_std > 0:
                frame_tensor = (
                    frame_tensor + torch.randn_like(frame_tensor) * noise_std
                )

            frame_tensor = torch.clamp(frame_tensor, 0.0, 1.0)

        return frame_tensor.float(), torch.tensor(label, dtype=torch.long)


# ===============================
# MODEL
# ===============================
class SpeakerCNN(nn.Module):
    """
    Minimal CNN for full-frame laryngoscopy classification.

    3 conv blocks  : 1 → 8 → 16 → 32 channels
    GAP + GMP      : [B, 32] + [B, 32] → [B, 64]
    Classifier     : Linear(64→32) → ReLU → Dropout(0.5) → Linear(32→4)

    Total parameters: ~8,200
    """
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 — low-level edges and tissue boundaries
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.GroupNorm(4, 8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # → [B,  8, 96, 128]

            # Block 2 — structural patterns
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # → [B, 16, 48,  64]

            # Block 3 — higher-level anatomy and texture
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # → [B, 32, 24,  32]
        )

        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        x   = self.features(x)
        gap = nn.functional.adaptive_avg_pool2d(x, 1).view(x.size(0), -1)  # [B, 32]
        gmp = nn.functional.adaptive_max_pool2d(x, 1).view(x.size(0), -1)  # [B, 32]
        x   = torch.cat([gap, gmp], dim=1)                                  # [B, 64]
        return self.classifier(x)


# ===============================
# TRAIN / EVAL
# ===============================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (out.argmax(dim=1) == y).sum().item()
        total      += y.size(0)

    return total_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader, criterion, target_names, print_report=True):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)
            total_loss += criterion(out, y).item()
            all_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))

    if print_report:
        print("\n" + "=" * 60)
        print("Classification Report")
        print("=" * 60)
        print(classification_report(
            all_labels, all_preds,
            target_names=target_names, digits=4,
        ))

        cm = confusion_matrix(all_labels, all_preds)
        print("Confusion Matrix:")
        print(cm)

        plt.figure(figsize=(6, 5))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names,
            yticklabels=target_names,
        )
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title("Confusion Matrix")
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_ROOT, "confusion_matrix.png"), dpi=300)
        plt.show()

        print(f"\nTest Loss:     {avg_loss:.4f}")
        print(f"Test Accuracy: {acc:.2f}%")

    return avg_loss, acc


# ===============================
# MAIN
# ===============================
if __name__ == "__main__":
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    # ── Quick folder diagnostic ────────────────────────────────────────
    print("\n── Folder diagnostic ──────────────────────────────────────")
    for label, dirs in [("TRAIN", TRAIN_DIRS), ("TEST", TEST_DIRS)]:
        for path in dirs:
            if os.path.isdir(path):
                folders = sorted(os.listdir(path))
                print(f"{label} ({path}):")
                for f in folders:
                    subj = extract_subject_name(f)
                    print(f"  '{f}'  →  subject='{subj}'")
            else:
                print(f"{label}: PATH NOT FOUND → {path}")
    print("───────────────────────────────────────────────────────────\n")

    if not mount_and_check(TRAIN_DIRS, TEST_DIRS):
        print("Exiting due to path issues.")
    else:
        # ── Build datasets ─────────────────────────────────────────────
        train_ds = SpeakerRandomFrameDataset(
            root_dirs=TRAIN_DIRS,              # ← list of dirs
            subject_to_idx=None,
            training=True,
            videos_per_subject=TRAIN_VIDEOS_PER_SUBJECT,
            video_selection_mode="first",
            random_seed=RANDOM_SEED,
        )

        test_ds = SpeakerRandomFrameDataset(
            root_dirs=TEST_DIRS,               # ← list of dirs
            subject_to_idx=train_ds.subject_to_idx,   # reuse train mapping
            training=False,
            videos_per_subject=TEST_VIDEOS_PER_SUBJECT,
            video_selection_mode=TEST_VIDEO_SELECTION_MODE,
            random_seed=RANDOM_SEED,
        )

        num_classes  = len(train_ds.subject_to_idx)
        target_names = [train_ds.idx_to_subject[i] for i in range(num_classes)]

        print(f"\nSubject mapping  : {train_ds.subject_to_idx}")
        print(f"Number of classes: {num_classes}")

        assert num_classes == 4, (
            f"Expected 4 subjects but found {num_classes}: "
            f"{list(train_ds.subject_to_idx.keys())}\n"
            "Check that your directories contain folders for all 4 subjects."
        )

        # ── DataLoaders ────────────────────────────────────────────────
        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=2, pin_memory=torch.cuda.is_available(),
        )
        test_loader = DataLoader(
            test_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=torch.cuda.is_available(),
        )

        # ── Model, optimiser, scheduler ────────────────────────────────
        model     = SpeakerCNN(num_classes=num_classes).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\nTotal trainable parameters : {total_params:,}")
        print(f"Training samples           : {len(train_ds)}")
        print(f"Parameters per sample      : {total_params / len(train_ds):.1f}")



        # ── REPLACE WITH THIS ──────────────────────────────────────────
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5,
            patience=4, min_lr=1e-6,
        )

        # ── Paths ──────────────────────────────────────────────────────
        best_model_path = os.path.join(SAVE_ROOT, "best_speaker_cnn.pth")
        history_path    = os.path.join(SAVE_ROOT, "training_history.npy")
        summary_path    = os.path.join(SAVE_ROOT, "summary.json")
        loss_plot_path  = os.path.join(SAVE_ROOT, "loss_plot.png")
        acc_plot_path   = os.path.join(SAVE_ROOT, "accuracy_plot.png")
        lr_plot_path    = os.path.join(SAVE_ROOT, "lr_schedule.png")

        # ── Training loop ──────────────────────────────────────────────
        best_test_loss    = float("inf")
        best_test_acc     = 0.0
        best_epoch        = -1
        epochs_no_improve = 0

        epoch_list         = []
        train_loss_history = []
        test_loss_history  = []
        train_acc_history  = []
        test_acc_history   = []
        lr_history         = []

        for epoch in range(EPOCHS):
            print("\n" + "=" * 60)
            print(f"Epoch {epoch + 1}/{EPOCHS}  |  "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e}")
            print("=" * 60)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            train_loss, train_acc = train_epoch(
                model, train_loader, optimizer, criterion
            )
            test_loss, test_acc = evaluate(
                model, test_loader, criterion,
                target_names=target_names, print_report=False,
            )

            print(f"Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.2f}%")
            print(f"Test  Loss: {test_loss:.4f}  |  Test  Acc: {test_acc:.2f}%")

            epoch_list.append(epoch + 1)
            train_loss_history.append(train_loss)
            test_loss_history.append(test_loss)
            train_acc_history.append(train_acc)
            test_acc_history.append(test_acc)
            lr_history.append(optimizer.param_groups[0]["lr"])

            scheduler.step(test_loss)

            # ── Early stopping on test loss ────────────────────────────
            if test_loss < best_test_loss:
                best_test_loss    = test_loss
                best_test_acc     = test_acc
                best_epoch        = epoch + 1
                epochs_no_improve = 0
                torch.save(model.state_dict(), best_model_path)
                print(f"  ✔ New best model — epoch {best_epoch}, "
                      f"test loss {best_test_loss:.4f}, "
                      f"test acc  {best_test_acc:.2f}%")
            else:
                epochs_no_improve += 1
                print(f"  No improvement for "
                      f"{epochs_no_improve}/{EARLY_STOPPING_PATIENCE} epochs.")

            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print(f"\nEarly stopping triggered at epoch {epoch + 1}.")
                break

        # ── Final evaluation ───────────────────────────────────────────
        print(f"\nBest epoch            : {best_epoch}")
        print(f"Best test loss        : {best_test_loss:.4f}")
        print(f"Test acc at best loss : {best_test_acc:.2f}%")

        if os.path.exists(best_model_path):
            model.load_state_dict(
                torch.load(best_model_path, map_location=device)
            )
            print(f"Loaded best model from '{best_model_path}'")

        print("\nFinal Evaluation on Test Set (Best Model)")
        final_test_loss, final_test_acc = evaluate(
            model, test_loader, criterion,
            target_names=target_names, print_report=True,
        )

        # ── Save history ───────────────────────────────────────────────
        history = {
            "epoch_list"         : epoch_list,
            "train_loss_history" : train_loss_history,
            "test_loss_history"  : test_loss_history,
            "train_acc_history"  : train_acc_history,
            "test_acc_history"   : test_acc_history,
            "lr_history"         : lr_history,
            "best_epoch"         : best_epoch,
            "best_test_loss"     : best_test_loss,
            "best_test_acc"      : best_test_acc,
            "final_test_loss"    : final_test_loss,
            "final_test_accuracy": final_test_acc,
            "subject_to_idx"     : train_ds.subject_to_idx,
            "config": {
                "TRAIN_DIRS"               : TRAIN_DIRS,
                "TEST_DIRS"                : TEST_DIRS,
                "IMG_HEIGHT"               : IMG_HEIGHT,
                "IMG_WIDTH"                : IMG_WIDTH,
                "BATCH_SIZE"               : BATCH_SIZE,
                "EPOCHS"                   : EPOCHS,
                "LR"                       : LR,
                "WEIGHT_DECAY"             : WEIGHT_DECAY,
                "AUG_PROB"                 : AUG_PROB,
                "ROTATION_RANGE"           : ROTATION_RANGE,
                "SCALE_RANGE"              : SCALE_RANGE,
                "BRIGHTNESS_RANGE"         : BRIGHTNESS_RANGE,
                "CONTRAST_RANGE"           : CONTRAST_RANGE,
                "NOISE_STD_RANGE"          : NOISE_STD_RANGE,
                "TRAIN_VIDEOS_PER_SUBJECT" : TRAIN_VIDEOS_PER_SUBJECT,
                "TEST_VIDEOS_PER_SUBJECT"  : TEST_VIDEOS_PER_SUBJECT,
                "TEST_VIDEO_SELECTION_MODE": TEST_VIDEO_SELECTION_MODE,
                "RANDOM_SEED"              : RANDOM_SEED,
                "EARLY_STOPPING_PATIENCE"  : EARLY_STOPPING_PATIENCE,
            },
        }

        np.save(history_path, history, allow_pickle=True)
        with open(summary_path, "w") as f:
            json.dump(history, f, indent=4)
        print(f"\nHistory saved to '{history_path}'")
        print(f"Summary  saved to '{summary_path}'")

        # ── Plots ──────────────────────────────────────────────────────
        def _plot(y1, y2, label1, label2, ylabel, title, path):
            plt.figure(figsize=(7, 5))
            plt.plot(epoch_list, y1, marker="o", label=label1)
            plt.plot(epoch_list, y2, marker="s", label=label2)
            plt.xlabel("Epoch")
            plt.ylabel(ylabel)
            plt.title(title)
            plt.xticks(epoch_list)
            plt.grid(True)
            plt.legend()
            plt.tight_layout()
            plt.savefig(path, dpi=300, bbox_inches="tight")
            plt.show()

        _plot(
            train_loss_history, test_loss_history,
            "Train Loss", "Test Loss",
            "Loss", "Loss Curves", loss_plot_path,
        )
        _plot(
            train_acc_history, test_acc_history,
            "Train Accuracy", "Test Accuracy",
            "Accuracy (%)", "Accuracy Curves", acc_plot_path,
        )

        plt.figure(figsize=(7, 4))
        plt.plot(epoch_list, lr_history, marker="o", color="purple")
        plt.xlabel("Epoch")
        plt.ylabel("Learning Rate")
        plt.title("LR Schedule")
        plt.xticks(epoch_list)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(lr_plot_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("\nAll done.")
        print(f"Saved outputs under: {SAVE_ROOT}")